# 面试问题：Tree of Thoughts 与普通 CoT 有什么区别？怎样实现生成、评估、剪枝、回溯和预算控制？

**一句话回答**：CoT 通常沿单一路径自回归，Tree of Thoughts 把中间“思路”视为可扩展状态，同时探索多个分支，用 evaluator 排序、去重、剪枝并可回溯。它把推理变成显式搜索，但质量取决于状态表示、候选覆盖、评价可信度和 test-time compute 预算。

本 Notebook 以 24 点为可验证环境，用 `Fraction` 手写 thought expansion、状态签名、beam search、精确 verifier、预算/停止和轨迹审计，不调用通用搜索库或 LLM。


In [ ]:
from dataclasses import dataclass  # 导入本单元所需的依赖。
from fractions import Fraction  # 导入本单元所需的依赖。
import math  # 导入本单元所需的依赖。

TARGET136=Fraction(24); START136=(4,7,8,8)  # 计算并保存当前步骤的中间状态。
assert TARGET136==24  # 用受控断言验证关键不变量。
assert Fraction(1,3)*3==1  # 用受控断言验证关键不变量。
assert len(START136)==4  # 用受控断言验证关键不变量。


## 1. Thought 必须落成可判定状态

自由文本难以去重和验证。24 点状态包含剩余精确数值及其表达式，深度每次减少一个操作数；`signature` 只看排序后的 Fraction，用于把不同措辞但等价的中间状态合并。生产任务也应提取结构化事实、约束和未完成子目标。


In [ ]:
@dataclass(frozen=True)  # 为下方定义附加声明式配置。
class State136:  # 定义承载本节状态与行为的数据结构。
    items:tuple; trace:tuple=()  # 计算并保存当前步骤的中间状态。
    @property  # 为下方定义附加声明式配置。
    def signature(self): return tuple(sorted((x[0] for x in self.items)))  # 定义本节可复用的核心函数。
init136=State136(tuple((Fraction(x),str(x)) for x in START136))  # 计算并保存当前步骤的中间状态。
assert len(init136.items)==4  # 用受控断言验证关键不变量。
assert init136.signature==(Fraction(4),Fraction(7),Fraction(8),Fraction(8))  # 用受控断言验证关键不变量。
assert not init136.trace  # 用受控断言验证关键不变量。


## 2. Generator 负责覆盖合法下一步，不负责宣布正确

枚举两个操作数和 `+,-,*,/`，对交换对称操作去重，对减除保留方向，除数为零跳过。LLM 场景中 generator 可提出若干 thought，但仍需 schema、去重与动作白名单；生成数量是 branching factor 的直接成本来源。


In [ ]:
def expand136(state):  # 定义本节可复用的核心函数。
    out={}  # 计算并保存当前步骤的中间状态。
    items=state.items  # 计算并保存当前步骤的中间状态。
    for i in range(len(items)):  # 遍历输入元素以累积或检查结果。
        for j in range(i+1,len(items)):  # 遍历输入元素以累积或检查结果。
            (a,ea),(b,eb)=items[i],items[j]; rest=[items[k] for k in range(len(items)) if k not in (i,j)]  # 计算并保存当前步骤的中间状态。
            candidates=[(a+b,f"({ea}+{eb})"),(a*b,f"({ea}*{eb})"),(a-b,f"({ea}-{eb})"),(b-a,f"({eb}-{ea})")]  # 计算并保存当前步骤的中间状态。
            if b: candidates.append((a/b,f"({ea}/{eb})"))  # 按当前条件选择后续控制路径。
            if a: candidates.append((b/a,f"({eb}/{ea})"))  # 按当前条件选择后续控制路径。
            for val,expr in candidates:  # 遍历输入元素以累积或检查结果。
                ns=State136(tuple(rest+[(val,expr)]),state.trace+(expr,)); out.setdefault(ns.signature,ns)  # 计算并保存当前步骤的中间状态。
    return list(out.values())  # 返回当前分支计算出的结果。
first136=expand136(init136)  # 计算并保存当前步骤的中间状态。
assert first136  # 用受控断言验证关键不变量。
assert all(len(s.items)==3 for s in first136)  # 用受控断言验证关键不变量。
assert len({s.signature for s in first136})==len(first136)  # 用受控断言验证关键不变量。


## 3. Evaluator 是排序启发式，不应冒充终局 verifier

非终局状态无法仅凭“某个数接近 24”判断一定可解。启发式用于预算内优先展开，例如考虑离目标距离、剩余深度和整数性；最终正确性必须由独立确定性 verifier 检查。错误 evaluator 会剪掉唯一正确分支，是 ToT 的主要失败模式。


In [ ]:
def heuristic136(state):  # 定义本节可复用的核心函数。
    distance=min(abs(float(v-TARGET136)) for v,_ in state.items)  # 计算并保存当前步骤的中间状态。
    fraction_penalty=sum(v.denominator!=1 for v,_ in state.items)*.1  # 计算并保存当前步骤的中间状态。
    return distance+.05*(len(state.items)-1)+fraction_penalty  # 返回当前分支计算出的结果。
ranked_first136=sorted(first136,key=heuristic136)  # 计算并保存当前步骤的中间状态。
assert heuristic136(ranked_first136[0])<=heuristic136(ranked_first136[-1])  # 用受控断言验证关键不变量。
assert all(math.isfinite(heuristic136(s)) for s in first136)  # 用受控断言验证关键不变量。
assert heuristic136(init136)>=0  # 用受控断言验证关键不变量。


## 4. Beam Search 在每层保留多个候选并允许间接回溯

每一深度展开 frontier，按 signature 全局去重，再保留 top-W；若高分分支后续失败，下一轮仍可从其他候选继续。beam width=1 类似贪心，较大宽度提高覆盖但线性增加 evaluator 与生成成本。


In [ ]:
def verify136(state): return len(state.items)==1 and state.items[0][0]==TARGET136  # 定义本节可复用的核心函数。
def beam_search136(start,width=100,budget=1000):  # 定义本节可复用的核心函数。
    frontier=[start]; seen={start.signature}; expanded=0  # 计算并保存当前步骤的中间状态。
    while frontier and expanded<budget:  # 在终止条件满足前持续推进状态。
        for s in frontier:  # 遍历输入元素以累积或检查结果。
            if verify136(s): return s,{"expanded":expanded,"seen":len(seen)}  # 按当前条件选择后续控制路径。
        candidates=[]  # 计算并保存当前步骤的中间状态。
        for s in frontier:  # 遍历输入元素以累积或检查结果。
            for ns in expand136(s):  # 遍历输入元素以累积或检查结果。
                expanded+=1  # 计算并保存当前步骤的中间状态。
                if ns.signature not in seen: seen.add(ns.signature); candidates.append(ns)  # 按当前条件选择后续控制路径。
                if expanded>=budget: break  # 按当前条件选择后续控制路径。
            if expanded>=budget: break  # 按当前条件选择后续控制路径。
        frontier=sorted(candidates,key=heuristic136)[:width]  # 计算并保存当前步骤的中间状态。
    return None,{"expanded":expanded,"seen":len(seen)}  # 返回当前分支计算出的结果。
solved136,stats136=beam_search136(init136,200,3000)  # 计算并保存当前步骤的中间状态。
assert solved136 is not None and verify136(solved136)  # 用受控断言验证关键不变量。
assert stats136["expanded"]<=3000  # 用受控断言验证关键不变量。
assert solved136.items[0][1]  # 用受控断言验证关键不变量。


## 5. Transposition table 把树收敛成状态图

不同操作顺序可能到达相同剩余数字集合；若环境满足 Markov 性，可只保留一个状态，显著减少重复扩展。若历史影响权限、成本或可用工具，则 signature 必须包含这些状态，不能仅按文本或最终数字错误合并。


In [ ]:
a136=State136(((Fraction(3),"(7-4)"),(Fraction(8),"8"),(Fraction(8),"8")),("7-4",))  # 计算并保存当前步骤的中间状态。
b136=State136(((Fraction(8),"8"),(Fraction(3),"three"),(Fraction(8),"8")),("alternate",))  # 计算并保存当前步骤的中间状态。
assert a136.signature==b136.signature  # 用受控断言验证关键不变量。
assert a136.trace!=b136.trace  # 用受控断言验证关键不变量。
assert len({a136.signature,b136.signature})==1  # 用受控断言验证关键不变量。


## 6. Final verifier 检查结果、约束和资源使用

24 点不仅要求结果为 24，还要求每个输入恰用一次且只用允许操作。教学 trace 由生成器保证资源约束，最终用 Fraction 避免浮点误判。开放任务则需单测、引用核验、schema 或环境反馈，不应让同一个 LLM 自评后直接提交高风险动作。


In [ ]:
assert verify136(State136(((Fraction(24),"((7-(8/8))*4)"),)))  # 用受控断言验证关键不变量。
assert not verify136(State136(((Fraction(23),"23"),)))  # 用受控断言验证关键不变量。
assert not verify136(State136(((Fraction(24),"24"),(Fraction(1),"unused"))))  # 用受控断言验证关键不变量。


## 7. Test-time compute 用边际收益和硬预算控制

预算可按候选数、模型 token、工具调用、wall clock 或金额度量。达到 verifier、deadline、最大深度、frontier 为空或连续扩展无改进即停止。高风险任务优先加 verifier 预算，而不是无上限扩大 generator beam。


In [ ]:
widths136=[1,5,50,200]; runs136=[beam_search136(init136,w,3000) for w in widths136]  # 计算并保存当前步骤的中间状态。
success136=[s is not None for s,_ in runs136]; costs136=[st["expanded"] for _,st in runs136]  # 计算并保存当前步骤的中间状态。
assert any(success136)  # 用受控断言验证关键不变量。
assert all(c<=3000 for c in costs136)  # 用受控断言验证关键不变量。
assert len(success136)==len(widths136)  # 用受控断言验证关键不变量。


## 8. 记录搜索决策才能定位“没生成”还是“被剪掉”

Trace 至少包含 node ID、parent、generator 版本、候选、evaluator score、prune reason、预算与 verifier 结果。离线做 generator oracle recall、evaluator pair accuracy、solution rate、nodes/tokens/latency；分别替换 generator、evaluator、width 才能归因。


In [ ]:
audit136={"start":START136,"width":200,"budget":3000,"expanded":stats136["expanded"],"solution":solved136.items[0][1],"verified":verify136(solved136)}  # 计算并保存当前步骤的中间状态。
assert audit136["verified"]  # 用受控断言验证关键不变量。
assert audit136["expanded"]>0  # 用受控断言验证关键不变量。
assert audit136["solution"].count("8")==2  # 用受控断言验证关键不变量。


## 面试总结

回答链路是：**把 thought 变成结构化 state → generator 枚举合法候选 → signature 去重 → evaluator 只排序 → beam/DFS/MCTS 在硬预算内探索 → 允许回溯 → 独立 verifier 判终局 → 记录生成/评分/剪枝 trace → 分别评测 generator recall、evaluator accuracy 和 solution/cost**。ToT 适合需要探索和可验证中间状态的任务，不应默认替代简单 CoT。

延伸阅读：[Tree of Thoughts](https://arxiv.org/abs/2305.10601)、[ReAct](https://arxiv.org/abs/2210.03629)、[RAP](https://arxiv.org/abs/2305.14992)。
